<a href="https://colab.research.google.com/github/Mohamed-Heni25/fraud-detection-job-offers/blob/main/nvda_pred_23/11/26_to_07/01/27_SentimentVsTsForcast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q finnhub-python yfinance transformers chronos-forecasting xgboost scikit-learn torch pandas pytz


Mounted at /content/drive


In [7]:
import finnhub
import pandas as pd
import time
from datetime import datetime, timedelta

FINNHUB_KEY = "dc28p6pr01qjj5fhmkj0dc28p6pr01qjj5fhmkjg"
finnhub_client = finnhub.Client(api_key=FINNHUB_KEY)

end_date = datetime.now().date()
start_date = end_date - timedelta(days=365)

all_articles = []
curr_start = start_date

while curr_start < end_date:
    curr_end = min(curr_start + timedelta(days=7), end_date)
    news = finnhub_client.company_news(
        "NVDA",
        _from=curr_start.strftime("%Y-%m-%d"),
        to=curr_end.strftime("%Y-%m-%d")
    )
    all_articles.extend(news)
    curr_start = curr_end + timedelta(days=1)
    time.sleep(1.1)  # Stay safely under Finnhub's 60 calls/min limit

news_df = pd.DataFrame(all_articles).drop_duplicates(subset=["id"])

# Keep headlines specifically focused on NVIDIA
mask = news_df["headline"].str.contains("NVIDIA|NVDA", case=False, na=False)
news_df = news_df[mask].copy()

# Convert UTC timestamps to US Eastern Time & shift after-close news (>= 4:00 PM ET)
dt_et = pd.to_datetime(news_df["datetime"], unit="s", utc=True).dt.tz_convert("America/New_York")
after_close = dt_et.dt.hour >= 16
dt_et = dt_et.where(~after_close, dt_et + pd.Timedelta(days=1))

# Roll Saturday (5) -> Monday (+2 days) and Sunday (6) -> Monday (+1 day)
dow = dt_et.dt.dayofweek
dt_et = dt_et.where(dow != 5, dt_et + pd.Timedelta(days=2))
dt_et = dt_et.where(dow != 6, dt_et + pd.Timedelta(days=1))

news_df["trade_date"] = dt_et.dt.date
print(f"Cleaned & aligned {len(news_df)} NVDA-specific headlines.")


Cleaned & aligned 2912 NVDA-specific headlines.


In [8]:
import torch
import yfinance as yf
from transformers import pipeline

sentiment_pipe = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    top_k=None,
    truncation=True,
    device=0 if torch.cuda.is_available() else -1
)

headlines = news_df["headline"].fillna("").tolist()
batch_results = sentiment_pipe(headlines, batch_size=32)

news_df["net_sentiment"] = [
    {item["label"]: item["score"] for item in res}["positive"] -
    {item["label"]: item["score"] for item in res}["negative"]
    for res in batch_results
]

daily_sentiment = news_df.groupby("trade_date").agg(
    sentiment_mean=("net_sentiment", "mean"),
    sentiment_std=("net_sentiment", "std"),
    article_count=("id", "count")
).reset_index()
daily_sentiment["trade_date"] = pd.to_datetime(daily_sentiment["trade_date"])

# Fetch matching NVDA daily prices
nvda = yf.download("NVDA", start=start_date, end=end_date, interval="1d", progress=False)
price_df = nvda[["Close", "Volume"]].reset_index()
price_df.columns = ["timestamp", "close", "volume"]
price_df["timestamp"] = pd.to_datetime(price_df["timestamp"]).dt.tz_localize(None)

df = pd.merge(price_df, daily_sentiment, left_on="timestamp", right_on="trade_date", how="left").drop(columns=["trade_date"])
df[["sentiment_mean", "sentiment_std", "article_count"]] = df[["sentiment_mean", "sentiment_std", "article_count"]].fillna(0.0)

# Save permanently to Google Drive
SAVE_PATH = "/content/drive/MyDrive/nvda_price_and_sentiment.csv"
df.to_csv(SAVE_PATH, index=False)
print(f"Saved dataset ({len(df)} trading days) to {SAVE_PATH}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_445/3425640941.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nvda = yf.download("NVDA", start=start_date, end=end_date, interval="1d", progress=False)


Saved dataset (250 trading days) to /content/drive/MyDrive/nvda_price_and_sentiment.csv


In [13]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("/content/drive/MyDrive/nvda_price_and_sentiment.csv")

# Engineer stationary features (using past data only—no leakage)
df["ret_1d"] = df["close"].pct_change(1)
df["ret_5d"] = df["close"].pct_change(5)
df["vol_10d"] = df["ret_1d"].rolling(10).std()
df["vol_ratio"] = df["volume"] / df["volume"].rolling(20).mean()
df["sent_3d_avg"] = df["sentiment_mean"].rolling(3).mean()

# Target: Next trading day's percentage return
df["target_next_ret"] = df["close"].pct_change(1).shift(-1)
df = df.dropna().reset_index(drop=True)

price_features = ["ret_1d", "ret_5d", "vol_10d", "vol_ratio"]
hybrid_features = price_features + ["sentiment_mean", "sentiment_std", "article_count", "sent_3d_avg"]

# Walk-forward validation across the final 60 trading days
test_days = 60
preds_price, preds_hybrid, actuals = [], [], []

for i in range(len(df) - test_days, len(df)):
    train = df.iloc[:i]
    test = df.iloc[i:i+1]

    m_price = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
    m_price.fit(train[price_features], train["target_next_ret"])
    preds_price.append(m_price.predict(test[price_features])[0])

    m_hybrid = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
    m_hybrid.fit(train[hybrid_features], train["target_next_ret"])
    preds_hybrid.append(m_hybrid.predict(test[hybrid_features])[0])

    actuals.append(test["target_next_ret"].values[0])

print(f"Price-Only MAE:       {mean_absolute_error(actuals, preds_price):.5f}")
print(f"Price + Sentiment MAE:{mean_absolute_error(actuals, preds_hybrid):.5f}")


Price-Only MAE:       0.03003
Price + Sentiment MAE:0.02880


In [21]:
import pandas as pd
import torch
from chronos import BaseChronosPipeline

raw_df = pd.read_csv("/content/drive/MyDrive/nvda_price_and_sentiment.csv")
raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"])
raw_df = raw_df.sort_values("timestamp").drop_duplicates(subset=["timestamp"])

# 1. Reindex onto a strict Monday–Friday business-day grid ("B") so Chronos sees uniform spacing
full_b_days = pd.date_range(start=raw_df["timestamp"].min(), end=raw_df["timestamp"].max(), freq="B")
chronos_df = raw_df.set_index("timestamp").reindex(full_b_days)

# Forward-fill closing prices on market holidays; set holiday news/sentiment to 0
chronos_df["close"] = chronos_df["close"].ffill()
chronos_df["sentiment_mean"] = chronos_df["sentiment_mean"].fillna(0.0)
chronos_df["article_count"] = chronos_df["article_count"].fillna(0.0)

chronos_df = chronos_df[["close", "sentiment_mean", "article_count"]].reset_index()
chronos_df = chronos_df.rename(columns={"index": "timestamp", "close": "target"})
chronos_df["item_id"] = "NVDA"

# 2. Load Chronos-2 & pass freq="B" explicitly
chronos_pipe = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cuda" if torch.cuda.is_available() else "cpu"
)

forecast = chronos_pipe.predict_df(
    chronos_df,
    prediction_length=10,
    quantile_levels=[0.3, 0.5, 0.6],
    id_column="item_id",
    timestamp_column="timestamp",
    target="target",
    freq="B",
)

print(forecast[["timestamp", "0.3", "0.5", "0.6"]])


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

   timestamp         0.3         0.5         0.6
0 2026-11-24  155.126358  161.469803  164.470367
1 2026-11-25  153.882904  161.772278  165.444077
2 2026-11-26  152.445892  161.718658  165.715714
3 2026-11-27  152.104919  161.868103  166.634872
4 2026-11-30  151.222839  161.863083  166.612198
5 2026-12-01  150.796234  161.543213  166.796524
6 2026-12-02  150.361526  161.724960  167.366226
7 2026-12-03  150.140686  162.152039  167.881927
8 2026-12-04  149.861465  162.283569  168.115585
9 2026-12-07  149.396103  162.498993  168.675171


In [20]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

# 1. Check feature importances on the final hybrid model
importances = pd.Series(m_hybrid.feature_importances_, index=hybrid_features).sort_values(ascending=False)
print("=== Top Feature Importances ===")
print(importances, "\n")

# 2. Strict Lag Test: Shift all sentiment columns back by 1 extra day (t-1)
# If MAE stays better than 0.03003 even with a 1-day buffer, your edge is rock-solid
df_strict = df.copy()
sent_cols = ["sentiment_mean", "sentiment_std", "article_count", "sent_3d_avg"]
df_strict[sent_cols] = df_strict[sent_cols].shift(1)
df_strict = df_strict.dropna().reset_index(drop=True)

preds_strict, actuals_strict = [], []
for i in range(len(df_strict) - 60, len(df_strict)):
    train = df_strict.iloc[:i]
    test = df_strict.iloc[i:i+1]
    m = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
    m.fit(train[hybrid_features], train["target_next_ret"])
    preds_strict.append(m.predict(test[hybrid_features])[0])
    actuals_strict.append(test["target_next_ret"].values[0])

print(f"Strict 1-Day Lag Sentiment MAE: {mean_absolute_error(actuals_strict, preds_strict):.5f}")


=== Top Feature Importances ===
article_count     0.171380
ret_1d            0.137863
ret_5d            0.131251
sentiment_std     0.119255
sent_3d_avg       0.114093
vol_ratio         0.112856
vol_10d           0.109282
sentiment_mean    0.104019
dtype: float32 

Strict 1-Day Lag Sentiment MAE: 0.02968
